In [1]:
import ast
import numpy as np
import pandas as pd
import wfdb
from sklearn.preprocessing import MultiLabelBinarizer

In [5]:
#consts 
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in (Path.cwd(), *Path.cwd().parents)
                            if (p / "scripts" / "project.py").exists()) / "scripts"))
from project import *  # ROOT, CACHE_FILE, LEAD_SET, MODELS_DIR, RESULTS_DIR, load_cache, ...
DATA_PATH = f"{DATASET_DIR.as_posix()}/"
SAMPLING_RATE = 100
LEAD_INDICES = [0, 1, 2]  # Lead I, II, III
TEST_FOLD = 10

In [6]:
def load_data(data_path=DATA_PATH):
    # 1. Load metadata and diagnostic statements
    df = pd.read_csv(f"{data_path}ptbxl_database.csv", index_col="ecg_id")
    df.scp_codes = df.scp_codes.apply(lambda x: ast.literal_eval(x))
    
    agg_df = pd.read_csv(f"{data_path}scp_statements.csv", index_col=0)
    agg_df = agg_df[agg_df.diagnostic == 1]

    def aggregate_diagnostic(y_dic):
        classes = []
        for key in y_dic.keys():
            if key in agg_df.index:
                classes.append(agg_df.loc[key].diagnostic_class)
        return list(set(classes))

    df["diagnostic_superclass"] = df.scp_codes.apply(aggregate_diagnostic)
    
    # Filter records containing at least one valid diagnostic superclass
    valid_mask = df["diagnostic_superclass"].apply(lambda x: len(x) > 0)
    df = df[valid_mask]

    # 2. Load low-resolution (100 Hz) signals
    signals = []
    for filename in df.filename_lr:
        record, _ = wfdb.rdsamp(f"{data_path}{filename}")
        signals.append(record[:, LEAD_INDICES])
    X = np.array(signals, dtype=np.float32)  # Shape: (N, 1000, 3)

    # 3. Multi-label binarization
    mlb = MultiLabelBinarizer()
    y = mlb.fit_transform(df["diagnostic_superclass"])
    classes = mlb.classes_

    # 4. Stratified Split (Fold 10 = Test)
    train_mask = (df.strat_fold != TEST_FOLD).values
    test_mask = (df.strat_fold == TEST_FOLD).values

    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    return X_train, y_train, X_test, y_test, classes

In [7]:
X_train, y_train, X_test, y_test, class_names = load_data()
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")
print(f"Target Superclasses: {class_names}")

Train Shape: (19230, 1000, 3), Test Shape: (2158, 1000, 3)
Target Superclasses: ['CD' 'HYP' 'MI' 'NORM' 'STTC']


In [ ]:
#cache/compress to avoid waiting time
import numpy as np

CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

save_dict = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
    "y_test": y_test,
    "class_names": np.array(class_names)
}

np.savez_compressed(CACHE_FILE, **save_dict)
print(f"Saved successfully to {CACHE_FILE}!")

Saving processed arrays and features to compressed .npz archive...
Saved successfully to ptbxl_preprocessed_cache.npz!
